# Text-to-SQL con LoRA - Entrenamiento Completo

Este notebook implementa el flujo completo del proyecto:
1. Preparación de datos
2. Evaluación del modelo base (baseline)
3. Entrenamiento con LoRA (configuración recomendada)
4. Evaluación del modelo fine-tuneado
5. Análisis de errores

**Autor**: Hansel López (19026)  
**Fecha**: 2026-01-25

## 1. Configuración del Entorno

In [ ]:
# Instalar dependencias
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q torch torchvision torchaudio
!pip install -q sentencepiece protobuf

print("✓ Dependencias instaladas")

In [ ]:
# Montar Google Drive para persistencia
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/text-to-sql-lora'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"✓ Directorio de trabajo: {WORK_DIR}")

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
import pandas as pd
from tqdm import tqdm
import re

# Verificar GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Usando dispositivo: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Preparación de Datos

In [ ]:
# Cargar dataset de entrenamiento
print("Cargando dataset de entrenamiento...")
with open('datos/entrenamiento/dataset_completo.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

print(f"✓ {len(train_data)} ejemplos cargados")

# Estadísticas por nivel
from collections import Counter
levels = Counter([item['nivel'] for item in train_data])
print(f"\nDistribución por nivel:")
for level, count in sorted(levels.items()):
    print(f"  {level}: {count} ({count/len(train_data)*100:.1f}%)")

In [ ]:
# Convertir a formato HuggingFace
def format_example(example):
    """Formatea un ejemplo para el modelo."""
    schema = example['schema']
    question = example['pregunta']
    sql = example['sql']
    
    prompt = f"""### Schema:
{schema}

### Question:
{question}

### SQL Query:
{sql}"""
    
    return {'text': prompt}

# Procesar dataset
processed_data = [format_example(item) for item in train_data]
dataset = Dataset.from_list(processed_data)

# Split train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"✓ Dataset creado: {len(dataset['train'])} train, {len(dataset['test'])} validation")

## 3. Evaluación del Modelo Base (Baseline)

In [ ]:
# Cargar modelo base
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

print(f"Cargando modelo base: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("✓ Modelo base cargado")

In [ ]:
# Cargar dataset de validación
with open('datos/validacion/validacion.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

print(f"✓ {len(val_data)} esquemas de validación cargados")

total_questions = sum(len(schema['questions']) for schema in val_data)
print(f"✓ {total_questions} preguntas de validación")

In [ ]:
def generate_sql_baseline(schema, question, model, tokenizer, max_new_tokens=256):
    """Genera SQL usando el modelo base."""
    prompt = f"""### Schema:
{schema}

### Question:
{question}

### SQL Query:
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraer solo la consulta SQL
    if "### SQL Query:" in response:
        sql = response.split("### SQL Query:")[-1].strip()
        # Tomar solo la primera línea o hasta el siguiente ###
        sql = sql.split("\n")[0].split("###")[0].strip()
        return sql
    
    return response.strip()

# Evaluar baseline
print("Evaluando modelo base (baseline)...\n")

baseline_results = []

for schema_data in tqdm(val_data, desc="Esquemas"):
    schema_name = schema_data['name']
    schema_sql = schema_data['esquema']
    
    for question_data in schema_data['questions']:
        question = question_data['question']
        reference_sql = question_data['sql']
        difficulty = question_data['level']
        
        # Generar predicción
        predicted_sql = generate_sql_baseline(schema_sql, question, model, tokenizer)
        
        baseline_results.append({
            'schema': schema_name,
            'question': question,
            'difficulty': difficulty,
            'reference_sql': reference_sql,
            'predicted_sql': predicted_sql,
            'exact_match': predicted_sql.strip().lower() == reference_sql.strip().lower()
        })

# Calcular métricas
baseline_df = pd.DataFrame(baseline_results)
baseline_accuracy = baseline_df['exact_match'].mean()

print(f"\n{'='*60}")
print("RESULTADOS BASELINE")
print(f"{'='*60}")
print(f"Exactitud total: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")

# Por nivel de dificultad
print("\nPor nivel de dificultad:")
for difficulty in ['facil', 'media', 'dificil']:
    mask = baseline_df['difficulty'] == difficulty
    if mask.sum() > 0:
        acc = baseline_df[mask]['exact_match'].mean()
        count = mask.sum()
        print(f"  {difficulty}: {acc:.4f} ({acc*100:.2f}%) [{count} preguntas]")

In [ ]:
# Guardar resultados baseline
baseline_df.to_csv('resultados/baseline_results.csv', index=False)
print("✓ Resultados baseline guardados en resultados/baseline_results.csv")

# Liberar memoria
del model
torch.cuda.empty_cache()

## 4. Entrenamiento con LoRA

In [ ]:
# Configuración LoRA (basada en análisis previo)
LORA_CONFIG = {
    "r": 16,                      # Rank
    "lora_alpha": 32,             # Alpha
    "lora_dropout": 0.05,         # Dropout
    "bias": "none",
    "task_type": "CAUSAL_LM",
    "target_modules": [            # Módulos a fine-tunear
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
}

print("Configuración LoRA:")
for key, value in LORA_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Recargar modelo para LoRA
print("Preparando modelo para LoRA...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Preparar para k-bit training
model = prepare_model_for_kbit_training(model)

# Aplicar LoRA
lora_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print("✓ Modelo configurado con LoRA")

In [ ]:
# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_32bit",
    max_grad_norm=0.3,
    report_to="none"
)

print("Configuración de entrenamiento:")
print(f"  Épocas: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")

In [ ]:
# Inicializar Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    tokenizer=tokenizer
)

# Entrenar
print("\nIniciando entrenamiento...\n")
train_result = trainer.train()

print("\n" + "="*60)
print("ENTRENAMIENTO COMPLETADO")
print("="*60)
print(f"Loss final: {train_result.training_loss:.4f}")
print(f"Épocas completadas: {train_result.metrics['train_epochs']}")

In [ ]:
# Guardar adapter
adapter_path = "adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"✓ Adapter guardado en {adapter_path}")

# Guardar también en Drive
drive_adapter_path = f"{WORK_DIR}/adapter"
model.save_pretrained(drive_adapter_path)
tokenizer.save_pretrained(drive_adapter_path)

print(f"✓ Adapter guardado en Drive: {drive_adapter_path}")

## 5. Evaluación del Modelo Fine-Tuneado

In [ ]:
# Cargar modelo con adapter
from peft import PeftModel

print("Cargando modelo base...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Cargando adapter LoRA...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()  # Merge para inferencia más rápida

print("✓ Modelo fine-tuneado cargado")

In [ ]:
def generate_sql_finetuned(schema, question, model, tokenizer, max_new_tokens=256):
    """Genera SQL usando el modelo fine-tuneado."""
    prompt = f"""### Schema:
{schema}

### Question:
{question}

### SQL Query:
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "### SQL Query:" in response:
        sql = response.split("### SQL Query:")[-1].strip()
        sql = sql.split("\n")[0].split("###")[0].strip()
        return sql
    
    return response.strip()

# Evaluar modelo fine-tuneado
print("Evaluando modelo fine-tuneado...\n")

finetuned_results = []

for schema_data in tqdm(val_data, desc="Esquemas"):
    schema_name = schema_data['name']
    schema_sql = schema_data['esquema']
    
    for question_data in schema_data['questions']:
        question = question_data['question']
        reference_sql = question_data['sql']
        difficulty = question_data['level']
        
        predicted_sql = generate_sql_finetuned(schema_sql, question, model, tokenizer)
        
        finetuned_results.append({
            'schema': schema_name,
            'question': question,
            'difficulty': difficulty,
            'reference_sql': reference_sql,
            'predicted_sql': predicted_sql,
            'exact_match': predicted_sql.strip().lower() == reference_sql.strip().lower()
        })

# Calcular métricas
finetuned_df = pd.DataFrame(finetuned_results)
finetuned_accuracy = finetuned_df['exact_match'].mean()

print(f"\n{'='*60}")
print("RESULTADOS MODELO FINE-TUNEADO")
print(f"{'='*60}")
print(f"Exactitud total: {finetuned_accuracy:.4f} ({finetuned_accuracy*100:.2f}%)")

print("\nPor nivel de dificultad:")
for difficulty in ['facil', 'media', 'dificil']:
    mask = finetuned_df['difficulty'] == difficulty
    if mask.sum() > 0:
        acc = finetuned_df[mask]['exact_match'].mean()
        count = mask.sum()
        print(f"  {difficulty}: {acc:.4f} ({acc*100:.2f}%) [{count} preguntas]")

In [ ]:
# Guardar resultados
finetuned_df.to_csv('resultados/finetuned_results.csv', index=False)
print("✓ Resultados fine-tuned guardados en resultados/finetuned_results.csv")

## 6. Comparación y Análisis de Errores

In [ ]:
# Comparar baseline vs fine-tuned
print("="*60)
print("COMPARACIÓN: BASELINE vs FINE-TUNED")
print("="*60)

improvement = finetuned_accuracy - baseline_accuracy
improvement_pct = (improvement / baseline_accuracy * 100) if baseline_accuracy > 0 else 0

print(f"\nBaseline:     {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"Fine-tuned:   {finetuned_accuracy:.4f} ({finetuned_accuracy*100:.2f}%)")
print(f"Mejora:       {improvement:+.4f} ({improvement_pct:+.1f}%)")

# Comparación por nivel
print("\nComparación por nivel de dificultad:")
for difficulty in ['facil', 'media', 'dificil']:
    baseline_mask = baseline_df['difficulty'] == difficulty
    finetuned_mask = finetuned_df['difficulty'] == difficulty
    
    if baseline_mask.sum() > 0 and finetuned_mask.sum() > 0:
        baseline_acc = baseline_df[baseline_mask]['exact_match'].mean()
        finetuned_acc = finetuned_df[finetuned_mask]['exact_match'].mean()
        diff = finetuned_acc - baseline_acc
        
        print(f"\n{difficulty.upper()}:")
        print(f"  Baseline:   {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
        print(f"  Fine-tuned: {finetuned_acc:.4f} ({finetuned_acc*100:.2f}%)")
        print(f"  Mejora:     {diff:+.4f} ({diff*100:+.1f}%)")

In [ ]:
# Análisis de errores
print("="*60)
print("ANÁLISIS DE ERRORES")
print("="*60)

# Preguntas que baseline falló pero fine-tuned acertó
improved = finetuned_df[
    (baseline_df['exact_match'] == False) & 
    (finetuned_df['exact_match'] == True)
]

print(f"\nPreguntas mejoradas (baseline→fine-tuned): {len(improved)}")
if len(improved) > 0:
    print("\nEjemplos:")
    for idx, row in improved.head(3).iterrows():
        print(f"\n  Pregunta: {row['question']}")
        print(f"  SQL referencia: {row['reference_sql'][:100]}...")
        print(f"  Baseline: {baseline_df.loc[idx, 'predicted_sql'][:100]}...")
        print(f"  Fine-tuned: {row['predicted_sql'][:100]}...")

# Preguntas que baseline acertó pero fine-tuned falló
regressed = finetuned_df[
    (baseline_df['exact_match'] == True) & 
    (finetuned_df['exact_match'] == False)
]

print(f"\n\nPreguntas regresivas (fine-tuned peor): {len(regressed)}")
if len(regressed) > 0:
    print("\nEjemplos:")
    for idx, row in regressed.head(3).iterrows():
        print(f"\n  Pregunta: {row['question']}")
        print(f"  SQL referencia: {row['reference_sql'][:100]}...")
        print(f"  Baseline: {baseline_df.loc[idx, 'predicted_sql'][:100]}...")
        print(f"  Fine-tuned: {row['predicted_sql'][:100]}...")

In [ ]:
# Guardar reporte final
report = {
    'baseline_accuracy': float(baseline_accuracy),
    'finetuned_accuracy': float(finetuned_accuracy),
    'improvement': float(improvement),
    'improvement_percentage': float(improvement_pct),
    'improved_questions': len(improved),
    'regressed_questions': len(regressed),
    'lora_config': LORA_CONFIG,
    'training_loss': float(train_result.training_loss),
    'epochs': float(train_result.metrics['train_epochs'])
}

with open('resultados/final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\n✓ Reporte final guardado en resultados/final_report.json")

print("\n" + "="*60)
print("PROYECTO COMPLETADO")
print("="*60)
print("\nArchivos generados:")
print("  - adapter/ (modelo LoRA)")
print("  - resultados/baseline_results.csv")
print("  - resultados/finetuned_results.csv")
print("  - resultados/final_report.json")

print("\nPróximos pasos:")
print("  1. Validar queries con MySQL (validar_queries.py)")
print("  2. Ejecutar evaluar.py con el adapter entrenado")
print("  3. Hacer commit y push a GitHub")